# BPR — Bayesian Personalized Ranking from Implicit Feedback (2009)

_Papers · Recommender Systems_

**Train a recommender on which item a user prefers over another, not on a rating number — so it learns to rank.**

---

This notebook is a practice scaffold for the **BPR — Bayesian Personalized Ranking from Implicit Feedback (2009)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn.functional as F
import math

torch.manual_seed(0)

# --- 0. Sanity-check the lesson's worked example. ---
w_u = torch.tensor([0.5, -0.2]); h_i = torch.tensor([0.4, 0.1]); h_j = torch.tensor([0.2, 0.3])
x_ui = torch.dot(w_u, h_i).item(); x_uj = torch.dot(w_u, h_j).item()
x_uij = x_ui - x_uj
sig   = 1.0 / (1.0 + math.exp(-x_uij))
grad  = (1.0 / (1.0 + math.exp(x_uij))) * (h_i - h_j)        # sigma(-x_uij) * (h_i - h_j)
print("worked example:  x_ui=%.4f x_uj=%.4f x_uij=%.4f" % (x_ui, x_uj, x_uij))
print("                 sigma(x_uij)=%.4f  ln sigma=%.4f" % (sig, math.log(sig)))
print("                 sigma(-x_uij)=%.4f  grad_w_u=%s" %
      (1/(1+math.exp(x_uij)), [round(v,4) for v in grad.tolist()]))
# worked example:  x_ui=0.1800 x_uj=0.0400 x_uij=0.1400
#                  sigma(x_uij)=0.5349  ln sigma=-0.6256
#                  sigma(-x_uij)=0.4651  grad_w_u=[0.093, -0.093]


# --- 1. A tiny synthetic implicit-feedback matrix with latent structure. ---
n_users, n_items, k = 30, 40, 8
g = torch.Generator().manual_seed(1)
U_true = torch.randn(n_users, 3, generator=g)
V_true = torch.randn(n_items, 3, generator=g)
S = U_true @ V_true.t()
R = torch.zeros(n_users, n_items)
for u in range(n_users):
    R[u, torch.topk(S[u], 6).indices] = 1.0          # ~6 positives per user

# Leave-one-out: hold out one positive per user for the test set (Section 6.2).
g2 = torch.Generator().manual_seed(2)
pos = {u: (R[u] > 0).nonzero().flatten().tolist() for u in range(n_users)}
test_pos, train_pos = {}, {}
for u in range(n_users):
    items = pos[u]
    ho = items[torch.randint(len(items), (1,), generator=g2).item()]
    test_pos[u]  = ho
    train_pos[u] = [i for i in items if i != ho]


def auc(W, H):                                        # Eqn. 2, Section 6.2
    with torch.no_grad():
        X, tot, cnt = W @ H.t(), 0.0, 0
        for u in range(n_users):
            i = test_pos[u]
            seen = set(train_pos[u]) | {i}
            negs = [j for j in range(n_items) if j not in seen and R[u, j] == 0]
            if not negs: continue
            tot += sum(1 for j in negs if X[u, i] > X[u, j]) / len(negs); cnt += 1
        return tot / cnt


# --- 2. LearnBPR: bootstrap-sampling SGD on the pairwise BPR-Opt loss (Fig. 4). ---
def sample_neg(u, gen):
    while True:
        j = torch.randint(n_items, (1,), generator=gen).item()
        if R[u, j] == 0: return j                     # j must be non-observed

def train_bpr(steps=4000, lr=0.1, lam=0.01):
    W = (0.1 * torch.randn(n_users, k)).requires_grad_(True)
    H = (0.1 * torch.randn(n_items, k)).requires_grad_(True)
    gen, curve = torch.Generator().manual_seed(7), []
    for t in range(steps + 1):
        if t % (steps // 12) == 0:
            curve.append((t, round(auc(W, H), 4)))
        u = torch.randint(n_users, (1,), generator=gen).item()
        if not train_pos[u]: continue
        i = train_pos[u][torch.randint(len(train_pos[u]), (1,), generator=gen).item()]
        j = sample_neg(u, gen)
        x_uij = (W[u] * (H[i] - H[j])).sum()          # x_ui - x_uj
        loss  = -F.logsigmoid(x_uij) + lam * ((W[u]**2).sum() + (H[i]**2).sum() + (H[j]**2).sum())
        loss.backward()
        with torch.no_grad():
            W -= lr * W.grad; H -= lr * H.grad; W.grad = None; H.grad = None
    return curve

# --- 3. Point-wise ABLATION: same model, but regress positives->1, negatives->0 (MSE). ---
def train_pointwise(steps=4000, lr=0.1, lam=0.01):
    W = (0.1 * torch.randn(n_users, k)).requires_grad_(True)
    H = (0.1 * torch.randn(n_items, k)).requires_grad_(True)
    gen, curve = torch.Generator().manual_seed(7), []
    for t in range(steps + 1):
        if t % (steps // 12) == 0:
            curve.append((t, round(auc(W, H), 4)))
        u = torch.randint(n_users, (1,), generator=gen).item()
        if not train_pos[u]: continue
        i = train_pos[u][torch.randint(len(train_pos[u]), (1,), generator=gen).item()]
        j = sample_neg(u, gen)
        x_ui = (W[u] * H[i]).sum(); x_uj = (W[u] * H[j]).sum()
        loss = (x_ui - 1.0)**2 + (x_uj - 0.0)**2 + lam * ((W[u]**2).sum() + (H[i]**2).sum() + (H[j]**2).sum())
        loss.backward()
        with torch.no_grad():
            W -= lr * W.grad; H -= lr * H.grad; W.grad = None; H.grad = None
    return curve

bpr  = train_bpr()
pw   = train_pointwise()
print("\nBPR-MF    AUC over training:", bpr)
print("Pointwise AUC over training:", pw)
print("BPR-MF: first AUC %.4f  ->  last AUC %.4f" % (bpr[0][1], bpr[-1][1]))
# BPR-MF AUC climbs from ~0.57 toward ~0.92: the pairwise loss orders held-out positives
# above negatives. (On this *easy* toy matrix the point-wise baseline is competitive too --
# the paper's gap shows on large, sparse, noisy data; see the ablation note.)
# Exact numbers vary by hardware/seed; this is our small run, not the paper's reported result.

## Visualize the data & results

_Does the pairwise BPR-Opt loss order held-out positives above negatives — does AUC rise above 0.5 over training?_

In [ ]:
# Same script as the CODE cell; the printed AUC lists are the series plotted above.
# Seeds are fixed, so re-running the CODE cell regenerates these exact numbers.
#
# Printed by the CODE cell:
#   BPR-MF    AUC over training: [(0,0.5735),(333,0.6578),...,(3996,0.9176)]   # green
#   Pointwise AUC over training: [(0,0.5735),(333,0.8039),...,(3996,0.9441)]   # red
#
# Our small run, not the paper's reported result. BPR-MF AUC: ~0.57 -> ~0.92 (the effect).
# On this easy toy matrix the point-wise baseline is comparable (~0.94); the paper's gap
# appears at large/sparse/noisy scale, not here -- reported honestly.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. Keep the same MF model, data, dimension, and optimizer, but swap the pairwise
            BPR loss for a point-wise baseline: regress each observed positive's score toward $1$ and
            each sampled non-observed item's score toward $0$ (mean-squared error). Retrain and compare
            held-out AUC. What do you expect, and what does the comparison isolate?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Change only the loss: from -logsigmoid(x_ui - x_uj) to (x_ui - 1)^2 + (x_uj - 0)^2; keep $W$, $H$, $k$, learning rate, and the train/test split identical. — _An honest ablation changes exactly one thing &mdash; the training criterion &mdash; so any AUC gap is attributable to it._
- Retrain and read off held-out AUC for both. — _Same model and data; only the objective differs. This is the paper's core claim (&sect;6.3): "the importance of optimizing models for the right criterion."_
- Note the scale honestly: on a large, sparse, noisy dataset (as in the paper) the point-wise 0-target hurts and BPR wins clearly; on a tiny, clean toy matrix the two can be comparable. — _The harm of forcing future-relevant items toward 0 grows with how many such items there are &mdash; it is most damaging at real scale._

**Answer:** At the paper's scale (&sect;6.3, large/sparse/noisy data) the BPR pairwise version reaches
                 clearly higher held-out AUC: the point-wise baseline wastes capacity forcing non-observed items
                 toward $0$ &mdash; items we actually want to rank highly in the future. On a tiny clean toy
                 matrix (our CODEVIZ run) the gap is small or even reversed, because with few items the
                 0-target does little harm; honestly, we do not reproduce the paper's gap at that scale. What the
                 toy run does show is the asked-for effect: BPR's held-out AUC climbing from
                 $\approx 0.5$ to $\approx 0.92$. The ablation isolates the training criterion &mdash;
                 same model, same data, only the loss changes &mdash; which is the paper's thesis: optimize for
                 ranking, not a point-wise target.

</details>

**Problem 2.** In one LearnBPR step the model already ranks the pair confidently right:
            $\hat{x}_{ui} = 3.0$, $\hat{x}_{uj} = -2.0$. Compute $\hat{x}_{uij}$, the loss term
            $-\ln\sigma(\hat{x}_{uij})$, and the gradient weight $\sigma(-\hat{x}_{uij})$. What does the
            tiny gradient weight tell you about how LearnBPR allocates effort?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- $\hat{x}_{uij} = 3.0 - (-2.0) = 5.0$. — _Pairwise score is the difference of the two item scores (&sect;4.3.1)._
- $\sigma(5.0) = 1/(1+e^{-5}) \approx 0.9933$, so the loss term $-\ln(0.9933) \approx 0.0067$. — _A confidently correct pair contributes almost nothing to the loss._
- Gradient weight $\sigma(-5.0) = 1/(1+e^{5}) \approx 0.0067$. — _$\sigma(-\hat{x}_{uij}) = 1 - \sigma(\hat{x}_{uij})$; here it is tiny._

**Answer:** $\hat{x}_{uij} = 5.0$, loss term $\approx 0.0067$, gradient weight
                 $\sigma(-5.0) \approx 0.0067$. Because the pair is already ranked confidently correctly, both
                 the loss and the gradient are nearly zero &mdash; LearnBPR barely touches it. The sigmoid weight
                 automatically steers each step toward the pairs the model still gets wrong (where
                 $\hat{x}_{uij}$ is negative and $\sigma(-\hat{x}_{uij})$ is large), which is why bootstrap
                 SGD converges efficiently despite the huge number of pairs.

</details>

**Problem 3.** Your worked example had $w_u = [0.5,-0.2]$, $h_i = [0.4,0.1]$, $h_j = [0.2,0.3]$, giving
            $\hat{x}_{uij} = 0.14$ and gradient-for-$w_u$ $= [0.093,-0.093]$. Take one gradient-ascent
            step on $w_u$ with learning rate $\alpha = 1.0$ (ignore regularization). What is the new $w_u$, and
            does $\hat{x}_{uij}$ increase?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- $w_u^{\text{new}} = w_u + \alpha\,[0.093,-0.093] = [0.5+0.093,\; -0.2-0.093] = [0.593,\,-0.293]$. — _Gradient ascent adds the gradient (BPR-Opt is maximized); $\alpha = 1$ here for a clean number._
- Recompute: $\hat{x}_{ui} = 0.593\cdot0.4 + (-0.293)\cdot0.1 = 0.2372 - 0.0293 = 0.2079$; $\hat{x}_{uj} = 0.593\cdot0.2 + (-0.293)\cdot0.3 = 0.1186 - 0.0879 = 0.0307$. — _Apply the new factor to both items with the &sect;4.3.1 dot product._
- $\hat{x}_{uij}^{\text{new}} = 0.2079 - 0.0307 = 0.1772 \gt 0.14$. — _The pairwise score rose, so the positive is now ranked further above the negative._

**Answer:** New $w_u = [0.593,\,-0.293]$, and $\hat{x}_{uij}$ rises from $0.14$ to $\approx 0.177$.
                 One ascent step on BPR-Opt moved $w_u$ toward $h_i$ and away from $h_j$, widening the
                 positive-minus-negative score gap &mdash; the model got better at ranking this exact pair, which
                 is precisely what the criterion asks for.

</details>